# xml: html처럼 태그 기반으로 자료를 저장한 포멧
* xml parser를 통해서 str을 xml로 변환하는 작업이 필요
* xml로 변환이 되면 태그 기반으로 자료를 찾아서 정리
* 태그에서 자료를 추출할 때는 beautifulsoup이라는 라이브러리를 이용

# 전세대출 데이터 수집

In [1]:
import os
import time
import requests
import pandas as pd

In [2]:
url = "http://apis.data.go.kr/B551408/rent-loan-rate-info/rate-list"
service_key = "8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A=="
payload = dict(serviceKey=service_key, numOfRows=20, pageNo=1, dataType='json')
r = requests.get(url, payload)
print(r.url)
print(r.status_code)
response = r.json()

http://apis.data.go.kr/B551408/rent-loan-rate-info/rate-list?serviceKey=8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A%3D%3D&numOfRows=20&pageNo=1&dataType=json
200


In [3]:
len(response['body']['items'])

18

In [4]:
result = {}
for item in response['body']['items']:
    for key, value in item.items():
        result.setdefault(key, []).append(value)
df = pd.DataFrame(result)
df

,bssYmdStart,interest4_1,interest3_2,interest4_2,interest2_1,interest1_2,interest3_1,interest2_2,interest1_1,bssYmdEnd,organId,callCenter
0,20250310,0,0,0,0,0,0,0,0,20250316,산업은행,1588-1500
1,20250310,0,0,0,0,0,0,0,0,20250316,제주은행,1588-0079
2,20250310,0,0,0,0,0,0,0,0,20250316,SC은행,1588-1599
3,20250310,4.89,0,0,0,0,0,0,0,20250316,전북은행,1588-4477
4,20250310,0,0,0,0,0,0,0,0,20250316,수협은행,1588-1515
5,20250310,4.26,0,0,0,0,0,0,0,20250316,우리은행,1599-5000
6,20250310,4.64,0,0,0,0,0,0,0,20250316,경남은행,1588-8585
7,20250310,3.5,0,0,0,0,0,0,0,20250316,카카오뱅크,1599-3333
8,20250310,3.38,0,0,0,0,0,0,0,20250316,아이엠뱅크,1588-5050
9,20250310,4.18,0,0,0,0,0,0,0,20250316,하나은행,1599-1111


# xml로 가져오기

In [184]:
url = "http://apis.data.go.kr/B551408/rent-loan-rate-info/rate-list"
service_key = "8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A=="
payload = dict(serviceKey=service_key, numOfRows=20, pageNo=1, dataType='xml')
r = requests.get(url, payload)
print(r.url)
print(r.status_code)
response = r.text
response

http://apis.data.go.kr/B551408/rent-loan-rate-info/rate-list?serviceKey=8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A%3D%3D&numOfRows=20&pageNo=1&dataType=xml
200


'<response><header><resultCode>00</resultCode><resultMsg>정상</resultMsg></header><body><pageNo>1</pageNo><totalCount>18</totalCount><numOfRows>20</numOfRows><items><item><bssYmdStart>20250310</bssYmdStart><interest4_1>0</interest4_1><interest3_2>0</interest3_2><interest4_2>0</interest4_2><interest2_1>0</interest2_1><interest1_2>0</interest1_2><interest3_1>0</interest3_1><interest2_2>0</interest2_2><interest1_1>0</interest1_1><bssYmdEnd>20250316</bssYmdEnd><organId>산업은행</organId><callCenter>1588-1500</callCenter></item><item><bssYmdStart>20250310</bssYmdStart><interest4_1>0</interest4_1><interest3_2>0</interest3_2><interest4_2>0</interest4_2><interest2_1>0</interest2_1><interest1_2>0</interest1_2><interest3_1>0</interest3_1><interest2_2>0</interest2_2><interest1_1>0</interest1_1><bssYmdEnd>20250316</bssYmdEnd><organId>제주은행</organId><callCenter>1588-0079</callCenter></item><item><bssYmdStart>20250310</bssYmdStart><interest4_1>0</interest4_1><interest3_2>0</interest3_2><interest4_2>0</inte

In [181]:
response.header

AttributeError: 'str' object has no attribute 'header'

* xml로 데이터를 받으면 처음에는 단순 문자열로 받게 된다.
* 문자열을 xml 문서로 변환해야 함
* beautifulsoup을 이용해 변환
* beautifulsoup의 메서드인 select, select_one을 이용해 css 셀렉터 기반으로 데이터 가 위치한 태그를 찾아서 내용 추출

# BesutifulSoup
* https://www.crummy.com/software/BeautifulSoup/bs4/doc/ 참고
* pip install beautifulsoup4
* from bs4 import BeautifulSoup as bs
* find, find_all 함수: xml, html에서 태그 기반으로 내용을 찾음
* select, select_one 함수: xml, html에서 css selector기반으로 내용을 찾음
* find_all, select는 해당 태그나 css selector를 가진 부분을 모두 찾아서 list로 반환
* find, select_one은 여러 태그나 css selector 중에서 가장 먼저 나오는 태그/selector를 한 개만 찾아줌
* 찾아온 태그 .name: 태그 이름 반환
* 찾아온 태그 .text,.string: 태그 안쪽의 텍스트 반환

In [12]:
from bs4 import BeautifulSoup as bs

In [182]:
soup = bs(response, 'xml')
print(soup)

<?xml version="1.0" encoding="utf-8"?>
<response><header><resultCode>00</resultCode><resultMsg>정상</resultMsg></header><body><pageNo>1</pageNo><totalCount>18</totalCount><numOfRows>20</numOfRows><items><item><bssYmdStart>20250310</bssYmdStart><interest4_1>0</interest4_1><interest3_2>0</interest3_2><interest4_2>0</interest4_2><interest2_1>0</interest2_1><interest1_2>0</interest1_2><interest3_1>0</interest3_1><interest2_2>0</interest2_2><interest1_1>0</interest1_1><bssYmdEnd>20250316</bssYmdEnd><organId>산업은행</organId><callCenter>1588-1500</callCenter></item><item><bssYmdStart>20250310</bssYmdStart><interest4_1>0</interest4_1><interest3_2>0</interest3_2><interest4_2>0</interest4_2><interest2_1>0</interest2_1><interest1_2>0</interest1_2><interest3_1>0</interest3_1><interest2_2>0</interest2_2><interest1_1>0</interest1_1><bssYmdEnd>20250316</bssYmdEnd><organId>제주은행</organId><callCenter>1588-0079</callCenter></item><item><bssYmdStart>20250310</bssYmdStart><interest4_1>0</interest4_1><interest3

In [9]:
soup.select_one("resultCode")

<resultCode>00</resultCode>

In [42]:
for tag in items.select("item > interest:nth-child(2n)"):
    print(tag.name)

bssYmdStart
interest4_1
interest3_2
interest4_2
interest2_1
interest1_2
interest3_1
interest2_2
interest1_1
bssYmdEnd
organId
callCenter


In [43]:
items = soup.select_one('items')
bssYmdStart_list = items.select("bssYmdStart")
interest4_1_list = items.select("interest4_1")
interest3_2_list = items.select("interest3_2")
interest4_2_list = items.select("interest4_2")
interest2_1_list = items.select("interest2_1")
interest1_2_list = items.select("interest1_2")
interest3_1_list = items.select("interest3_1")
interest2_2_list = items.select("interest2_2")
interest1_1_list = items.select("interest1_1")
bssYmdEnd_list = items.select("bssYmdEnd")
organId_list = items.select("organId")
callCenter_list = items.select("callCenter")

In [49]:
tag_list = [bssYmdStart_list, interest4_1_list, interest3_2_list, interest4_2_list, interest2_1_list, interest1_2_list, interest3_1_list, interest2_2_list, interest1_1_list, bssYmdEnd_list, organId_list, callCenter_list]

In [50]:
result = {}
for lists in tag_list:
    for item in lists:
        result.setdefault(item.name, []).append(item.text)
result

{'bssYmdStart': ['20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310',
  '20250310'],
 'interest4_1': ['0',
  '0',
  '0',
  '4.89',
  '0',
  '4.26',
  '4.64',
  '3.5',
  '3.38',
  '4.18',
  '3.74',
  '3.7',
  '3.89',
  '3.84',
  '4.05',
  '4.0',
  '4.11',
  '4.07'],
 'interest3_2': ['0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0'],
 'interest4_2': ['0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0'],
 'interest2_1': ['0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0'],
 'interest1_2': ['0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0'

In [52]:
final_df = pd.DataFrame(result)
final_df

,bssYmdStart,interest4_1,interest3_2,interest4_2,interest2_1,interest1_2,interest3_1,interest2_2,interest1_1,bssYmdEnd,organId,callCenter
0,20250310,0,0,0,0,0,0,0,0,20250316,산업은행,1588-1500
1,20250310,0,0,0,0,0,0,0,0,20250316,제주은행,1588-0079
2,20250310,0,0,0,0,0,0,0,0,20250316,SC은행,1588-1599
3,20250310,4.89,0,0,0,0,0,0,0,20250316,전북은행,1588-4477
4,20250310,0,0,0,0,0,0,0,0,20250316,수협은행,1588-1515
5,20250310,4.26,0,0,0,0,0,0,0,20250316,우리은행,1599-5000
6,20250310,4.64,0,0,0,0,0,0,0,20250316,경남은행,1588-8585
7,20250310,3.5,0,0,0,0,0,0,0,20250316,카카오뱅크,1599-3333
8,20250310,3.38,0,0,0,0,0,0,0,20250316,아이엠뱅크,1588-5050
9,20250310,4.18,0,0,0,0,0,0,0,20250316,하나은행,1599-1111


# 반복문으로 태그를 자동 추출해서 만들기

In [83]:
for tag in soup.select("items")[0]:
    for key in tag:
        print(key.name, key.text)

bssYmdStart 20250310
interest4_1 0
interest3_2 0
interest4_2 0
interest2_1 0
interest1_2 0
interest3_1 0
interest2_2 0
interest1_1 0
bssYmdEnd 20250316
organId 산업은행
callCenter 1588-1500
bssYmdStart 20250310
interest4_1 0
interest3_2 0
interest4_2 0
interest2_1 0
interest1_2 0
interest3_1 0
interest2_2 0
interest1_1 0
bssYmdEnd 20250316
organId 제주은행
callCenter 1588-0079
bssYmdStart 20250310
interest4_1 0
interest3_2 0
interest4_2 0
interest2_1 0
interest1_2 0
interest3_1 0
interest2_2 0
interest1_1 0
bssYmdEnd 20250316
organId SC은행
callCenter 1588-1599
bssYmdStart 20250310
interest4_1 4.89
interest3_2 0
interest4_2 0
interest2_1 0
interest1_2 0
interest3_1 0
interest2_2 0
interest1_1 0
bssYmdEnd 20250316
organId 전북은행
callCenter 1588-4477
bssYmdStart 20250310
interest4_1 0
interest3_2 0
interest4_2 0
interest2_1 0
interest1_2 0
interest3_1 0
interest2_2 0
interest1_1 0
bssYmdEnd 20250316
organId 수협은행
callCenter 1588-1515
bssYmdStart 20250310
interest4_1 4.26
interest3_2 0
interest4_2 0
i

In [96]:
result = {}
for tags in soup.select("item"):
    for tag in tags:
        result.setdefault(tag.name, []).append(tag.text)
df = pd.DataFrame(result)
df

,bssYmdStart,interest4_1,interest3_2,interest4_2,interest2_1,interest1_2,interest3_1,interest2_2,interest1_1,bssYmdEnd,organId,callCenter
0,20250310,0,0,0,0,0,0,0,0,20250316,산업은행,1588-1500
1,20250310,0,0,0,0,0,0,0,0,20250316,제주은행,1588-0079
2,20250310,0,0,0,0,0,0,0,0,20250316,SC은행,1588-1599
3,20250310,4.89,0,0,0,0,0,0,0,20250316,전북은행,1588-4477
4,20250310,0,0,0,0,0,0,0,0,20250316,수협은행,1588-1515
5,20250310,4.26,0,0,0,0,0,0,0,20250316,우리은행,1599-5000
6,20250310,4.64,0,0,0,0,0,0,0,20250316,경남은행,1588-8585
7,20250310,3.5,0,0,0,0,0,0,0,20250316,카카오뱅크,1599-3333
8,20250310,3.38,0,0,0,0,0,0,0,20250316,아이엠뱅크,1588-5050
9,20250310,4.18,0,0,0,0,0,0,0,20250316,하나은행,1599-1111


In [94]:
soup.select("item")

[<item><bssYmdStart>20250310</bssYmdStart><interest4_1>0</interest4_1><interest3_2>0</interest3_2><interest4_2>0</interest4_2><interest2_1>0</interest2_1><interest1_2>0</interest1_2><interest3_1>0</interest3_1><interest2_2>0</interest2_2><interest1_1>0</interest1_1><bssYmdEnd>20250316</bssYmdEnd><organId>산업은행</organId><callCenter>1588-1500</callCenter></item>,
 <item><bssYmdStart>20250310</bssYmdStart><interest4_1>0</interest4_1><interest3_2>0</interest3_2><interest4_2>0</interest4_2><interest2_1>0</interest2_1><interest1_2>0</interest1_2><interest3_1>0</interest3_1><interest2_2>0</interest2_2><interest1_1>0</interest1_1><bssYmdEnd>20250316</bssYmdEnd><organId>제주은행</organId><callCenter>1588-0079</callCenter></item>,
 <item><bssYmdStart>20250310</bssYmdStart><interest4_1>0</interest4_1><interest3_2>0</interest3_2><interest4_2>0</interest4_2><interest2_1>0</interest2_1><interest1_2>0</interest1_2><interest3_1>0</interest3_1><interest2_2>0</interest2_2><interest1_1>0</interest1_1><bssYmdE

# xml에 있는 모든 key를 추출하기

In [92]:
tag_names = []
for items in soup.select("item"):
    for item in items:
        tag_names.append(item.name)
tag_names

['bssYmdStart',
 'interest4_1',
 'interest3_2',
 'interest4_2',
 'interest2_1',
 'interest1_2',
 'interest3_1',
 'interest2_2',
 'interest1_1',
 'bssYmdEnd',
 'organId',
 'callCenter',
 'bssYmdStart',
 'interest4_1',
 'interest3_2',
 'interest4_2',
 'interest2_1',
 'interest1_2',
 'interest3_1',
 'interest2_2',
 'interest1_1',
 'bssYmdEnd',
 'organId',
 'callCenter',
 'bssYmdStart',
 'interest4_1',
 'interest3_2',
 'interest4_2',
 'interest2_1',
 'interest1_2',
 'interest3_1',
 'interest2_2',
 'interest1_1',
 'bssYmdEnd',
 'organId',
 'callCenter',
 'bssYmdStart',
 'interest4_1',
 'interest3_2',
 'interest4_2',
 'interest2_1',
 'interest1_2',
 'interest3_1',
 'interest2_2',
 'interest1_1',
 'bssYmdEnd',
 'organId',
 'callCenter',
 'bssYmdStart',
 'interest4_1',
 'interest3_2',
 'interest4_2',
 'interest2_1',
 'interest1_2',
 'interest3_1',
 'interest2_2',
 'interest1_1',
 'bssYmdEnd',
 'organId',
 'callCenter',
 'bssYmdStart',
 'interest4_1',
 'interest3_2',
 'interest4_2',
 'interest2

# 집합자료형 set : 중복 없는 데이터 집합 만들기

In [93]:
tags = set(tag_names)
tags

{'bssYmdEnd',
 'bssYmdStart',
 'callCenter',
 'interest1_1',
 'interest1_2',
 'interest2_1',
 'interest2_2',
 'interest3_1',
 'interest3_2',
 'interest4_1',
 'interest4_2',
 'organId'}

# 공공데이터 서민대출 데이터 수집
* 금융위원회_서민금융상품기본정보


In [97]:
import os
import requests
import pandas as pd
from bs4 import BeautifulSoup as bs

# json 데이터 정리하기

In [118]:
result_list = []
page = 1
while True:
    url = "https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo"
    service_key = "8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A=="
    payload = dict(serviceKey=service_key, numOfRows=1000, pageNo=page, resultType='json')
    r = requests.get(url, params=payload)
    print(r.url)
    print(r.status_code)
    response = r.json()
    total_pages = response['response']['body']['totalCount'] //response['response']['body']['numOfRows'] + 1
    result = {}
    for item in response['response']['body']['items']['item']:
        for key, value in item.items():
            result.setdefault(key, []).append(value)
    df = pd.DataFrame(result)
    result_list.append(df)
    if page <  total_pages:
        page += 1
    else:
        break
        
result_df = pd.concat(result_list)
result_df = result_df.reset_index(drop=True)
result_df

https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo?serviceKey=8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A%3D%3D&numOfRows=1000&pageNo=1&resultType=json
200
https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo?serviceKey=8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A%3D%3D&numOfRows=1000&pageNo=2&resultType=json
200
https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo?serviceKey=8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A%3D%3D&numOfRows=1000&pageNo=3&resultType=json
200
https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo?serviceKey=8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A%3D%3D&numOfRows=1000&pageNo=4&resultType=json


,basYm,snq,finPrdNm,lnLmt,irtCtg,irt,maxTotLnTrm,maxDfrmTrm,maxRdptTrm,rdptMthd,...,prdExisYn,cv19Rfrc,prdCtg,prdNm,cv19SuprCtg,cv19SuprCtn,cv19SuprTgtDtlCtn,mgmDln,prdCtg2,fileWrtDt
0,202503,1,새희망홀씨Ⅱ,3500만원,변동금리,은행별 상이,은행별 상이,-,-,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
1,202503,2,징검다리론,3000만원,변동금리,은행별 상이,5년,1년,4년,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,기관 문의,-,202504010700
2,202503,3,"우리지역 氣-Up 서포트론(영세 소기업,소상공인)","5,00010,000 (코로나 19 피해 소기업, 소상공인)",변동금리,6.27~8.97%,5년,1년,4년,원금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
3,202503,4,경상남도 청년 전세자금(경상남도 협약 상품)(경상남도 청년주택 임차보증금 이자지원사업),9000만원,변동금리,은행별 상이,임대차계약기간 이내 최대 2년,0년,0년,일시상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
4,202503,5,i-ONE근로자생활안정자금대출,1000만원,고정금리,1.5 / 2.6 / 1.0 / 3.0,"근로자생활안정자금, 임금체불생계비, 체불근로자생계비 : 2년, 4년, 5년, 6년,...",6년,5년,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4736,202110,425,"온(溫, ON)택트 특례보증",2850만원,변동금리,대출실행 은행에 문의,5년,1년,4년,원금균등분할상환,...,Y,None,1,대출상품,None,None,None,None,None,202502010700
4737,202110,426,위기업종 지원 협약보증,5000만원,-,-,"1, 5년",1년,4년,원금균등분할상환,...,Y,None,1,대출상품,None,None,None,None,None,202502010700
4738,202110,427,익산시 소상공인 특례보증(2023 하나은행 출연),5000만원,고정금리,약 1.6%,5년,1년,4년,원금균등분할상환,...,Y,None,1,대출상품,None,None,None,None,None,202502010700
4739,202110,428,중소벤처기업부 소상공인자금,7000만원,자금별 별도금리 적용,-,5년,2년,3년,분할상환,...,Y,None,1,대출상품,None,None,None,None,None,202502010700


# XML 데이터 정리하기

In [140]:
xml_result_df_list = []
page = 1
while True:
    url = "https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo"
    service_key = "8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A=="
    payload = dict(serviceKey=service_key, numOfRows=1000, pageNo=page, resultType='xml')
    r = requests.get(url, params=payload)
    print(r.url)
    print(r.status_code)
    response = r.content
    soup = bs(response, "xml")
    xml_result = {}
    for item in soup.select("item"):
        for tags in item:
            if tags.name == None:
                continue
            else:
                xml_result.setdefault(tags.name, []).append(tags.text)
    xml_result_df_list.append(pd.DataFrame(xml_result))
    numOfRows = int(soup.select_one("numOfRows").text)
    totalCount = int(soup.select_one("totalCount").text)
    total_pages = totalCount // numOfRows + 1
    if page < total_pages:
        page += 1
    else:
        break
        
xml_result_df_list = pd.concat(xml_result_df_list)
xml_result_df_list = xml_result_df_list.reset_index(drop=True)
xml_result_df_list

https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo?serviceKey=8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A%3D%3D&numOfRows=1000&pageNo=1&resultType=xml
200
https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo?serviceKey=8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A%3D%3D&numOfRows=1000&pageNo=2&resultType=xml
200
https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo?serviceKey=8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A%3D%3D&numOfRows=1000&pageNo=3&resultType=xml
200
https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo?serviceKey=8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A%3D%3D&numOfRows=1000&pageNo=4&resultType=xml
200


,basYm,snq,finPrdNm,lnLmt,irtCtg,irt,maxTotLnTrm,maxDfrmTrm,maxRdptTrm,rdptMthd,...,prdExisYn,cv19Rfrc,prdCtg,prdNm,cv19SuprCtg,cv19SuprCtn,cv19SuprTgtDtlCtn,mgmDln,prdCtg2,fileWrtDt
0,202503,1,새희망홀씨Ⅱ,3500만원,변동금리,은행별 상이,은행별 상이,-,-,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
1,202503,2,징검다리론,3000만원,변동금리,은행별 상이,5년,1년,4년,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,기관 문의,-,202504010700
2,202503,3,"우리지역 氣-Up 서포트론(영세 소기업,소상공인)","5,00010,000 (코로나 19 피해 소기업, 소상공인)",변동금리,6.27~8.97%,5년,1년,4년,원금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
3,202503,4,경상남도 청년 전세자금(경상남도 협약 상품)(경상남도 청년주택 임차보증금 이자지원사업),9000만원,변동금리,은행별 상이,임대차계약기간 이내 최대 2년,0년,0년,일시상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
4,202503,5,i-ONE근로자생활안정자금대출,1000만원,고정금리,1.5 / 2.6 / 1.0 / 3.0,"근로자생활안정자금, 임금체불생계비, 체불근로자생계비 : 2년, 4년, 5년, 6년,...",6년,5년,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4736,202110,425,"온(溫, ON)택트 특례보증",2850만원,변동금리,대출실행 은행에 문의,5년,1년,4년,원금균등분할상환,...,Y,,1,대출상품,,,,,,202502010700
4737,202110,426,위기업종 지원 협약보증,5000만원,-,-,"1, 5년",1년,4년,원금균등분할상환,...,Y,,1,대출상품,,,,,,202502010700
4738,202110,427,익산시 소상공인 특례보증(2023 하나은행 출연),5000만원,고정금리,약 1.6%,5년,1년,4년,원금균등분할상환,...,Y,,1,대출상품,,,,,,202502010700
4739,202110,428,중소벤처기업부 소상공인자금,7000만원,자금별 별도금리 적용,-,5년,2년,3년,분할상환,...,Y,,1,대출상품,,,,,,202502010700


setdefault를 이용해 xml의 tag명의 유일값 추출하기

In [124]:
tag_names = {}
for tags in soup.select("*"):
    tag_names.setdefault(tags.name, "")
for tag_name in tag_names.keys():
    print(tag_name)
    

response
header
resultCode
resultMsg
body
numOfRows
pageNo
totalCount
items
item
basYm
snq
finPrdNm
lnLmt
irtCtg
irt
maxTotLnTrm
maxDfrmTrm
maxRdptTrm
rdptMthd
usge
trgt
instCtg
ofrInstNm
rsdAreaPamtEqltIstm
suprTgtDtlCond
age
incm
rsdArea
crdtSc
anin
housHoldCnt
housAr
lnTgtHous
rfrcCnpl
grnInst
jnMthd
rpymdCfe
lnIcdcst
ovItrYr
prftAddIrtCond
etcRefSbjc
hdlInst
cnpl
rltSite
tgtFltr
hdlInstDtlVw
prdExisYn
cv19Rfrc
prdCtg
prdNm
cv19SuprCtg
cv19SuprCtn
cv19SuprTgtDtlCtn
mgmDln
prdCtg2
fileWrtDt


In [138]:
int(soup.select_one("totalCount").text)

4741

# key_list를 만들고 select로 직접 추출하기

In [175]:
df_list = []
page = 1
while True:
    url = "https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo"
    service_key = "8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A=="
    payload = dict(serviceKey=service_key, numOfRows=1000, pageNo=page, resultType='xml')
    r = requests.get(url, params=payload)
    print(r.url)
    print(r.status_code)
    response = r.content
    soup = bs(response, "xml")
    numOfRows = int(soup.select_one("numOfRows").text)
    totalCount = int(soup.select_one("totalCount").text)
    total_pages = totalCount // numOfRows + 1
    key_list = {}
    for item in soup.select("*"):
        key_list.setdefault(item.name, "")
    keys = list(key_list.keys())
    result = {}
    for key in keys[10:]:
        for item in soup.select(key):
            result.setdefault(item.name, []).append(item.text)
    df_list.append(pd.DataFrame(result))
    if page < total_pages:
        page += 1
    else:
        print("데이터 수집완료")
        break
result_df = pd.concat(df_list)
result_df = result_df.reset_index(drop=True)
result_df

https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo?serviceKey=8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A%3D%3D&numOfRows=1000&pageNo=1&resultType=xml
200
https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo?serviceKey=8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A%3D%3D&numOfRows=1000&pageNo=2&resultType=xml
200
https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo?serviceKey=8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A%3D%3D&numOfRows=1000&pageNo=3&resultType=xml
200
https://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo?serviceKey=8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A%3D%3D&numOfRows=1000&pageNo=4&resultType=xml
200


,basYm,snq,finPrdNm,lnLmt,irtCtg,irt,maxTotLnTrm,maxDfrmTrm,maxRdptTrm,rdptMthd,...,prdExisYn,cv19Rfrc,prdCtg,prdNm,cv19SuprCtg,cv19SuprCtn,cv19SuprTgtDtlCtn,mgmDln,prdCtg2,fileWrtDt
0,202503,1,새희망홀씨Ⅱ,3500만원,변동금리,은행별 상이,은행별 상이,-,-,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
1,202503,2,징검다리론,3000만원,변동금리,은행별 상이,5년,1년,4년,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,기관 문의,-,202504010700
2,202503,3,"우리지역 氣-Up 서포트론(영세 소기업,소상공인)","5,00010,000 (코로나 19 피해 소기업, 소상공인)",변동금리,6.27~8.97%,5년,1년,4년,원금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
3,202503,4,경상남도 청년 전세자금(경상남도 협약 상품)(경상남도 청년주택 임차보증금 이자지원사업),9000만원,변동금리,은행별 상이,임대차계약기간 이내 최대 2년,0년,0년,일시상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
4,202503,5,i-ONE근로자생활안정자금대출,1000만원,고정금리,1.5 / 2.6 / 1.0 / 3.0,"근로자생활안정자금, 임금체불생계비, 체불근로자생계비 : 2년, 4년, 5년, 6년,...",6년,5년,원(리)금균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202504010700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4736,202110,425,"온(溫, ON)택트 특례보증",2850만원,변동금리,대출실행 은행에 문의,5년,1년,4년,원금균등분할상환,...,Y,,1,대출상품,,,,,,202502010700
4737,202110,426,위기업종 지원 협약보증,5000만원,-,-,"1, 5년",1년,4년,원금균등분할상환,...,Y,,1,대출상품,,,,,,202502010700
4738,202110,427,익산시 소상공인 특례보증(2023 하나은행 출연),5000만원,고정금리,약 1.6%,5년,1년,4년,원금균등분할상환,...,Y,,1,대출상품,,,,,,202502010700
4739,202110,428,중소벤처기업부 소상공인자금,7000만원,자금별 별도금리 적용,-,5년,2년,3년,분할상환,...,Y,,1,대출상품,,,,,,202502010700


In [168]:
keys

['response',
 'header',
 'resultCode',
 'resultMsg',
 'body',
 'numOfRows',
 'pageNo',
 'totalCount',
 'items',
 'item',
 'basYm',
 'snq',
 'finPrdNm',
 'lnLmt',
 'irtCtg',
 'irt',
 'maxTotLnTrm',
 'maxDfrmTrm',
 'maxRdptTrm',
 'rdptMthd',
 'usge',
 'trgt',
 'instCtg',
 'ofrInstNm',
 'rsdAreaPamtEqltIstm',
 'suprTgtDtlCond',
 'age',
 'incm',
 'rsdArea',
 'crdtSc',
 'anin',
 'housHoldCnt',
 'housAr',
 'lnTgtHous',
 'rfrcCnpl',
 'grnInst',
 'jnMthd',
 'rpymdCfe',
 'lnIcdcst',
 'ovItrYr',
 'prftAddIrtCond',
 'etcRefSbjc',
 'hdlInst',
 'cnpl',
 'rltSite',
 'tgtFltr',
 'hdlInstDtlVw',
 'prdExisYn',
 'cv19Rfrc',
 'prdCtg',
 'prdNm',
 'cv19SuprCtg',
 'cv19SuprCtn',
 'cv19SuprTgtDtlCtn',
 'mgmDln',
 'prdCtg2',
 'fileWrtDt']